In [ ]:
import polars as pl
import pandas as pd
import plotly.express as px
import numpy as np
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os
from pathlib import Path

CWD = str(Path.cwd())
if 'notebooks' in Path.cwd().name:
    CWD = str(Path.cwd().parent)
    sys.path.insert(1, CWD)



# Data Preparation

In [13]:
from src_strategy.configs.dataconfig import input_output_config_3_1

OUTPUT_DIR = Path(input_output_config_3_1.output_path)
INPUT_DIR = Path(input_output_config_3_1.data_path)
SIRS_PATH = OUTPUT_DIR / "df_sirs.parquet"
ENCOUNTER_PATH = INPUT_DIR / "Encounter Table with Baseline Values - June 2022 - May 2026 - 6.5.26.csv"
AGGREGATED_PATH = OUTPUT_DIR / "df_aggregated.parquet"
INFECTION_PATH = OUTPUT_DIR / "df_suspected_infection.parquet"
EVENT_PATH = OUTPUT_DIR / "df_all_no_collisions.parquet"

In [19]:
df_agg = pl.read_parquet(AGGREGATED_PATH)
df_all = pl.read_parquet(EVENT_PATH)
df_encs = pl.read_csv(ENCOUNTER_PATH)
if df_encs.schema['Arrival_Instant'] == pl.Utf8:
	df_encs = df_encs.with_columns(
		pl.col('Arrival_Instant').str.strptime(pl.Datetime, '%Y-%m-%d %H:%M:%S.%f', strict=False)
	)

/tmp/ipykernel_1833184/2318722121.py:6: ChronoFormatWarning: Detected the pattern `.%f` in the chrono format string. This pattern should not be used to parse values after a decimal point. Use `%.f` instead. See the full specification: https://docs.rs/chrono/latest/chrono/format/strftime
  pl.col('Arrival_Instant').str.strptime(pl.Datetime, '%Y-%m-%d %H:%M:%S.%f', strict=False)


In [25]:
[c for c in df_encs.columns if c.startswith('Baseline_')]

['Baseline_SBP',
 'Baseline_RespiratoryRate',
 'Baseline_PulseRate',
 'Baseline_Creatinine',
 'Baseline_Platelets',
 'Baseline_Bilirubin',
 'Baseline_eGFR',
 'Baseline_WBC']

In [31]:
df_bp = df_agg.select(
    "EncounterEpicCsn",
	"last_sbp_8h",
	"last_map_8h",
	"last_sbp_8h_source_ts",
    "last_map_8h_source_ts"
)

In [32]:
df_bp

EncounterEpicCsn,last_sbp_8h,last_map_8h,last_sbp_8h_source_ts,last_map_8h_source_ts
i64,f64,f64,datetime[μs],datetime[μs]
659308243,null,null,null,null
659308243,null,null,null,null
659308243,null,null,null,null
659308243,138.0,95.3,2022-06-14 11:52:00,2022-06-14 11:52:00
659308243,138.0,95.3,2022-06-14 11:52:00,2022-06-14 11:52:00
…,…,…,…,…
753774459,132.0,99.3,2026-05-28 15:00:00,2026-05-28 15:00:00
753774459,149.0,103.0,2026-05-28 19:27:00,2026-05-28 19:27:00
753774459,141.0,107.0,2026-05-28 22:28:00,2026-05-28 22:28:00


In [33]:
df_bp_with_baseline = df_bp.filter(
    pl.col("last_sbp_8h").is_not_null()
    |pl.col("last_map_8h").is_not_null() 
).join(
    df_encs.select(["EncounterEpicCsn", pl.col("Baseline_SBP").cast(pl.Float64, strict=False)]),
	on="EncounterEpicCsn",
	how="left"
)

In [35]:
df_bp_with_baseline

EncounterEpicCsn,last_sbp_8h,last_map_8h,last_sbp_8h_source_ts,last_map_8h_source_ts,Baseline_SBP
i64,f64,f64,datetime[μs],datetime[μs],f64
659308243,138.0,95.3,2022-06-14 11:52:00,2022-06-14 11:52:00,134.0
659308243,138.0,95.3,2022-06-14 11:52:00,2022-06-14 11:52:00,134.0
659308243,138.0,95.3,2022-06-14 11:52:00,2022-06-14 11:52:00,134.0
659308243,138.0,95.3,2022-06-14 11:52:00,2022-06-14 11:52:00,134.0
659308243,138.0,95.3,2022-06-14 11:52:00,2022-06-14 11:52:00,134.0
…,…,…,…,…,…
753774459,132.0,99.3,2026-05-28 15:00:00,2026-05-28 15:00:00,null
753774459,149.0,103.0,2026-05-28 19:27:00,2026-05-28 19:27:00,null
753774459,141.0,107.0,2026-05-28 22:28:00,2026-05-28 22:28:00,null


# Data Transformation

In [39]:
df_bp_with_baseline = df_bp_with_baseline.with_columns(
	pl.when(pl.col("Baseline_SBP").is_not_null()).then(
        ((pl.col("Baseline_SBP") -pl.col("last_sbp_8h")) > 40)
	).otherwise(
        pl.col("last_sbp_8h") < 90
	).alias("sbp_hypotension"),
    (pl.col("last_map_8h")<65).alias("map_hypotension")
).with_columns(
    (pl.col("map_hypotension")|pl.col("sbp_hypotension")).alias("hypotension")
)

In [45]:
df_bp_with_baseline['sbp_hypotension'].value_counts().with_columns(
    (100.0*pl.col("count")/pl.col("count").sum()).alias("%")
).sort(by="%")

sbp_hypotension,count,%
bool,u64,f64
true,330733,4.195009
null,1564408,19.842912
false,5988823,75.962079


In [46]:
df_bp_with_baseline['map_hypotension'].value_counts().with_columns(
    (100.0*pl.col("count")/pl.col("count").sum()).alias("%")
).sort(by="%")

map_hypotension,count,%
bool,u64,f64
true,797606,10.116814
false,7086358,89.883186
